In [5]:
from typing import List # list로 자료형을하면 뭐 모범답안을만들수있다?
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

"""
표준 정답지(Golden Set)
- 질문 (user_input)
- 모범정답 (reference)
- 근거문서 (retrieved_contexts)
"""
# ㄴ> (필드명) - 이름을지운이유가 라가스로평가할때 데이터셋의 어떤 항목 이름이 이런식으로 짓기위해

class RagEvalSample(BaseModel):
    user_input: str = Field(description="제공된 근거 문서들을 바탕으로 사용자가 질의할법한 질문")
    reference: str = Field(description="근거 문서에 기반한 모범정답(Ground Truth)")
    retrived_contesxts: List[str] = Field(
        description="해당 질문과 정답을 도출하는 데 실제로 사용된 1개 이상의 원본 문서 내용"
    )


class RagEvalDataset(BaseModel):
    samples: List[RagEvalSample] = Field(
        description="근거 문서로부터 생성된 평가 샘플 목록"
    )

source_docs: List[Document] = [
    Document(
        page_content="원격 근무 지원 규정: 주 2회 재택근무가 가능하며, 원격 근무 당일은 오전 9시까지 사내 메신저 근태 채널에 출근 등록을 완료해야 합니다.",
        metadata={"source": "근태규정.pdf", "chunk_id": 1}
    ),
    Document(
        page_content="복리후생 지원 규정: 재택근무 지원금은 월 50,000원이 지급되며, 매월 25일 급여일에 합산되어 지급됩니다.",
        metadata={"source": "복리후생.pdf", "chunk_id": 2}
    ),
    Document(
        page_content="수습 사원 근무 가이드: 신규 입사자는 입사 후 3개월간 수습 기간을 거치며, 수습 기간 중에는 원격 근무 및 관련 지원금 신청이 제한됩니다.",
        metadata={"source": "인사핸드북.pdf", "chunk_id": 3}
    ),
]

def format_documents(docs):
    _docs = [f"[문서 {doc.metadata.get("chunk_id")}]\n{doc.page_content.strip()}" for doc in source_docs]

    return "\n\n".join(_docs)

In [ ]:
model = init_chat_model(
    model="gpt-6-luna",
    model_provider="openai",
    reasoning_effort="low" # 온도값.? temparture..?랑 다른건지
)

structured_model = model.with_structured_output(RagEvalDataset)

prompt = PromptTemplate.from_template(
    """
    당신은 RAG 평가 데이터를 만드는 전문가입니다.
    제공된 여러 개의 [근거 문서 목록]을 분석하여 사용자가 질의할 법한 '질문'과 '정답' 쌍을 {num_samples}개 생성하세요.

    [작성 지침]
    질문은 다음과 같은 유형으로 구성하세요.
    - 단순 사실 확인 : 하나의 문서를 근거해서 단순하게 답변할 수 있는 질문
    - 복합 추론 질문 : 여러개의 문서를 근거하고 논리적 추론이 필요한 질문
    - 문서 외 함정 질문 : 제공된 문서에 등장하지 않는 근거가 필요한 질문(환각 확인 목적)

    [근거 문서 목록]
    {context}
    """
) #{..}안에잇는게 placeholder | 형식에 대한근거를 넣어도 좋다..

chain = prompt | structured_model

result = chain.invoke({
    "context": format_documents(source_docs),
    "num_samples": 4
})

result


RagEvalDataset(samples=[RagEvalSample(user_input='원격 근무는 주 몇 회까지 가능하며, 원격 근무 당일 출근 등록은 몇 시까지 해야 하나요?', reference='원격 근무는 주 2회 가능하며, 원격 근무 당일 오전 9시까지 사내 메신저 근태 채널에 출근 등록을 완료해야 합니다.', retrived_contesxts=['원격 근무 지원 규정: 주 2회 재택근무가 가능하며, 원격 근무 당일은 오전 9시까지 사내 메신저 근태 채널에 출근 등록을 완료해야 합니다.']), RagEvalSample(user_input='재택근무 지원금은 얼마이며, 언제 지급되나요?', reference='재택근무 지원금은 월 50,000원이며, 매월 25일 급여일에 급여와 합산되어 지급됩니다.', retrived_contesxts=['복리후생 지원 규정: 재택근무 지원금은 월 50,000원이 지급되며, 매월 25일 급여일에 합산되어 지급됩니다.']), RagEvalSample(user_input='입사한 지 2개월 된 신규 입사자가 재택근무를 하면서 지원금도 신청할 수 있나요?', reference='아니요. 신규 입사자는 입사 후 3개월간 수습 기간을 거치며, 수습 기간 중에는 원격 근무와 관련 지원금 신청이 모두 제한됩니다. 따라서 입사 2개월 차에는 재택근무를 할 수 없고 지원금도 신청할 수 없습니다.', retrived_contesxts=['수습 사원 근무 가이드: 신규 입사자는 입사 후 3개월간 수습 기간을 거치며, 수습 기간 중에는 원격 근무 및 관련 지원금 신청이 제한됩니다.', '원격 근무 지원 규정: 주 2회 재택근무가 가능하며, 원격 근무 당일은 오전 9시까지 사내 메신저 근태 채널에 출근 등록을 완료해야 합니다.', '복리후생 지원 규정: 재택근무 지원금은 월 50,000원이 지급되며, 매월 25일 급여일에 합산되어 지급됩니다.']), RagEvalSample(user_input='재택근무를 하는 직원은 회사에서 노트북이나 

In [ ]:
"""
RagEvalDataset(samples=[RagEvalSample(user_input='원격 근무는 주 몇 회까지 가능하며, 원격 근무 당일 출근 등록은 몇 시까지 해야 하나요?', reference='원격 근무는 주 2회 가능하며, 원격 근무 당일 오전 9시까지 사내 메신저 근태 채널에 출근 등록을 완료해야 합니다.', retrived_contesxts=['원격 근무 지원 규정: 주 2회 재택근무가 가능하며, 원격 근무 당일은 오전 9시까지 사내 메신저 근태 채널에 출근 등록을 완료해야 합니다.']), RagEvalSample(user_input='재택근무 지원금은 얼마이며, 언제 지급되나요?', reference='재택근무 지원금은 월 50,000원이며, 매월 25일 급여일에 급여와 합산되어 지급됩니다.', retrived_contesxts=['복리후생 지원 규정: 재택근무 지원금은 월 50,000원이 지급되며, 매월 25일 급여일에 합산되어 지급됩니다.']), RagEvalSample(user_input='입사한 지 2개월 된 신규 입사자가 재택근무를 하면서 지원금도 신청할 수 있나요?', reference='아니요. 신규 입사자는 입사 후 3개월간 수습 기간을 거치며, 수습 기간 중에는 원격 근무와 관련 지원금 신청이 모두 제한됩니다. 따라서 입사 2개월 차에는 재택근무를 할 수 없고 지원금도 신청할 수 없습니다.', retrived_contesxts=['수습 사원 근무 가이드: 신규 입사자는 입사 후 3개월간 수습 기간을 거치며, 수습 기간 중에는 원격 근무 및 관련 지원금 신청이 제한됩니다.', '원격 근무 지원 규정: 주 2회 재택근무가 가능하며, 원격 근무 당일은 오전 9시까지 사내 메신저 근태 채널에 출근 등록을 완료해야 합니다.', '복리후생 지원 규정: 재택근무 지원금은 월 50,000원이 지급되며, 매월 25일 급여일에 합산되어 지급됩니다.']), RagEvalSample(user_input='재택근무를 하는 직원은 회사에서 노트북이나 모니터를 대여받을 수 있나요?', reference='제공된 문서에는 재택근무 시 노트북이나 모니터를 대여할 수 있는지에 대한 내용이 없습니다.', retrived_contesxts=['원격 근무 지원 규정: 주 2회 재택근무가 가능하며, 원격 근무 당일은 오전 9시까지 사내 메신저 근태 채널에 출근 등록을 완료해야 합니다.', '복리후생 지원 규정: 재택근무 지원금은 월 50,000원이 지급되며, 매월 25일 급여일에 합산되어 지급됩니다.', '수습 사원 근무 가이드: 신규 입사자는 입사 후 3개월간 수습 기간을 거치며, 수습 기간 중에는 원격 근무 및 관련 지원금 신청이 제한됩니다.'])])
"""